# 06. OTM ma'lumotlarini tayyorlash (Data Preparation)

`05`-notebookdagi xulosalar asosida `final_otm_2026.csv` ni tozalaymiz:

1. `Unnamed: 0` ni olib tashlash
2. Takrorlarni o'chirish va kalit to'qnashuvini hal qilish
3. Kvotasi umuman yo'q qatorlarni olib tashlash
4. Kvota 0 bo'lgan joyda o'tish ballarini NaN ga almashtirish
5. `asosiy_yunalish`, `tuman_kvotasi`, `tuman` belgilarini yaratish
6. Fanlar majmuasidan `1-fan`, `2-fan`, `fanlar_juftligi` ni biriktirish
7. Tekshirish va `data/processed/otm_prepared.csv` ga saqlash

**Belgi va maqsad:** kvotalar, OTM, yo'nalish, til, shakl va fanlar qabuldan oldin ma'lum — belgi bo'la oladi. `grant_bali` va `shartnoma_bali` mandatdan keyin ma'lum bo'ladi — ular maqsadli o'zgaruvchi.

In [1]:
import re

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

In [2]:
df = pd.read_csv('../data/processed/final_otm_2026.csv')
print(df.shape)
df.head()

(5712, 10)


,Unnamed: 0,hudud,otm_nomi,yunalish_nomi,talim_tili,talim_shakli,grant_kvota_soni,shartnoma_kvota_soni,grant_bali,shartnoma_bali
0,0,Qoraqalpog‘iston Respublikasi,Nukus davlat pedagogika instituti,Amaliy matematika,O'zbekcha,Kunduzgi,2,20,165.9,110.3
1,1,Qoraqalpog‘iston Respublikasi,Nukus davlat pedagogika instituti,Amaliy matematika,O'zbekcha,Kechki,0,13,0.0,61.9
2,2,Qoraqalpog‘iston Respublikasi,Nukus davlat pedagogika instituti,Amaliy matematika,Русский,Kunduzgi,1,4,167.8,59.8
3,3,Qoraqalpog‘iston Respublikasi,Nukus davlat pedagogika instituti,Amaliy matematika,Qoraqalpoq,Kunduzgi,1,24,170.1,64.3
4,4,Qoraqalpog‘iston Respublikasi,Nukus davlat pedagogika instituti,Amaliy matematika,Qoraqalpoq,Kechki,0,12,0.0,56.9


## 1. `Unnamed: 0` ni olib tashlash

Bu CSV ga `index=False` siz saqlangan eski index.

In [3]:
df = df.drop(columns='Unnamed: 0')
df.shape

(5712, 9)

## 2. Takrorlar

Avval to'liq takrorlarni o'chiramiz, keyin `otm_nomi + yunalish_nomi + talim_tili + talim_shakli` kaliti bo'yicha qolgan to'qnashuvni ko'ramiz.

In [4]:
KALIT = ['otm_nomi', 'yunalish_nomi', 'talim_tili', 'talim_shakli']

print('To\'liq takrorlar:', df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

toqnashuv = df[df.duplicated(KALIT, keep=False)]
print('Kalit to\'qnashuvi:', len(toqnashuv), 'qator')
toqnashuv

To'liq takrorlar: 13
Kalit to'qnashuvi: 2 qator


,hudud,otm_nomi,yunalish_nomi,talim_tili,talim_shakli,grant_kvota_soni,shartnoma_kvota_soni,grant_bali,shartnoma_bali
5163,Toshkent shahri,Toshkent davlat iqtisodiyot universiteti,Davlat auditi,O'zbekcha,Kunduzgi,0,150,0.0,176.4
5164,Toshkent shahri,Toshkent davlat iqtisodiyot universiteti,Davlat auditi,O'zbekcha,Kunduzgi,0,9,0.0,176.4


TDIU `Davlat auditi` ikki qator bo'lib kelgan: o'tish bali bir xil (176.4), faqat shartnoma kvotasi farq qiladi (150 va 9). Ball bir xil bo'lgani uchun bu bitta tanlov guruhi deb hisoblaymiz: kvotalarni qo'shamiz, ballni saqlaymiz.

In [5]:
agg = {col: 'first' for col in df.columns if col not in KALIT}
agg.update({'grant_kvota_soni': 'sum', 'shartnoma_kvota_soni': 'sum',
            'grant_bali': 'max', 'shartnoma_bali': 'max'})

df = df.groupby(KALIT, as_index=False, sort=False).agg(agg)[df.columns]
assert not df.duplicated(KALIT).any()
df.shape

(5698, 9)

## 3. Kvotasi yo'q qatorlar

Grant ham, shartnoma ham kvotasi 0 — bu yo'nalish bu yil qabul qilmagan. Na belgi, na maqsad bor.

In [6]:
kvota_yoq = (df['grant_kvota_soni'] == 0) & (df['shartnoma_kvota_soni'] == 0)
print('Olib tashlanadi:', kvota_yoq.sum())
df = df[~kvota_yoq].reset_index(drop=True)
df.shape

Olib tashlanadi: 151


(5547, 9)

## 4. 0 ballarni NaN ga almashtirish

`05`-notebookda tasdiqlandi: kvota 0 ⇔ ball 0. 0 — "bu yil bu tur bo'yicha qabul bo'lmagan", haqiqiy ball emas. NaN bu ma'noni to'g'ri saqlaydi va o'rtacha kabi hisob-kitoblarni buzmaydi.

In [7]:
for tur in ['grant', 'shartnoma']:
    kvota, bal = f'{tur}_kvota_soni', f'{tur}_bali'
    assert ((df[kvota] == 0) == (df[bal] == 0)).all(), f'{tur}: kvota va ball mos emas'
    df.loc[df[kvota] == 0, bal] = np.nan

df[['grant_bali', 'shartnoma_bali']].describe().round(1)

,grant_bali,shartnoma_bali
count,3376.0,4718.0
mean,140.2,86.8
std,40.9,37.5
min,64.0,56.7
25%,106.8,57.9
50%,152.4,65.5
75%,175.7,108.9
max,191.5,187.9


## 5. Yo'nalish nomidan belgilar

Qavs ichida tuman maqsadli kvotasi (`Davolash ishi (Kitob tumani)`) yoki ixtisoslik (`Cholgʻu ijrochiligi: fortepiano (organ)`) bo'lishi mumkin.

In [8]:
qavs_ichi = df['yunalish_nomi'].str.extract(r'\(([^)]*)\)\s*$')[0]
df['tuman_kvotasi'] = qavs_ichi.str.contains(r'\b(?:tumani|shahri)\b', regex=True, na=False)
df['tuman'] = qavs_ichi.where(df['tuman_kvotasi'])
df['asosiy_yunalish'] = df['yunalish_nomi'].str.replace(r'\s*\([^)]*\)', '', regex=True).str.strip()

print('Tuman kvotasi:       ', df['tuman_kvotasi'].sum())
print('Noyob tumanlar:      ', df['tuman'].nunique())
print('Asosiy yo\'nalishlar:', df['asosiy_yunalish'].nunique())
df.loc[df['tuman_kvotasi'], ['yunalish_nomi', 'asosiy_yunalish', 'tuman']].head()

Tuman kvotasi:        724
Noyob tumanlar:       183
Asosiy yo'nalishlar: 341


,yunalish_nomi,asosiy_yunalish,tuman
5,Amaliy matematika (Ellikqalʼa tumani),Amaliy matematika,Ellikqalʼa tumani
6,Amaliy matematika (Moʻynoq tumani),Amaliy matematika,Moʻynoq tumani
24,Fizika (Ellikqalʼa tumani),Fizika,Ellikqalʼa tumani
50,Matematika (Ellikqalʼa tumani),Matematika,Ellikqalʼa tumani
59,"Milliy gʻoya, maʼnaviyat asoslari va huquq taʼlimi (Qoʻngʻirot tumani)","Milliy gʻoya, maʼnaviyat asoslari va huquq taʼlimi",Qoʻngʻirot tumani


## 6. Fanlar majmuasini biriktirish

Abituriyentlar jadvali OTM ga faqat fanlar juftligi orqali bog'lanadi, shuning uchun har bir yo'nalishga `1-fan` va `2-fan` kerak. To'rt bosqichda moslashtiramiz:

1. **Aniq moslik** — kichik harf, apostroflar bir xil, qavslar olib tashlangan nom bo'yicha.
2. **Imlo xatolari** — qo'lda tuzilgan lug'at.
3. **Qo'shimcha manba** — fanlar majmuasida yo'q yo'nalishlar uchun `kirish_imtihon_fanlari.md`.
4. **Asosiy qism** — `:` gacha bo'lgan qism bo'yicha (masalan, `Sport faoliyati: voleybol` → `Sport faoliyati`). Agar bu qismdagi yo'nalishlar turli fanlarga ega bo'lsa, eng ko'p uchraydigan juftlik olinadi (masalan, `Filologiya va tillarni oʻqitish: pushtu tili` → Chet tili + Ona tili).

Fuzzy moslashtirish ishlatilmaydi: u `pushtu tili` ni `rus tili` ga bog'laydi — fanlar noto'g'ri chiqadi.

In [9]:
fanlar = pd.read_excel('../data/raw/Fanlar_majmuasi_2025-2026.xlsx')
fanlar = fanlar.rename(columns={'Bakalavriat taʼlim yoʻnalishlari nomi': 'nom'})

# 04-notebookdagi kabi Kasbiy fan nomini bir xillashtiramiz
fan_tuzatish = {'Kasbiy (ijodiy) imtihon': 'Kasbiy (ijodiy imtihon)'}
fanlar[['1-fan', '2-fan']] = fanlar[['1-fan', '2-fan']].replace(fan_tuzatish)

def normallash(nom):
    nom = str(nom).lower()
    nom = re.sub(r"[ʻ‘’ʼ'`]", "'", nom)
    nom = re.sub(r'\s*\([^)]*\)', '', nom)
    return re.sub(r'\s+', ' ', nom).strip()

fanlar['kalit'] = fanlar['nom'].map(normallash)
assert fanlar['kalit'].is_unique
print('Fanlar majmuasidagi yo\'nalishlar:', len(fanlar))

Fanlar majmuasidagi yo'nalishlar: 218


### Qo'shimcha manba: `kirish_imtihon_fanlari.md`

Fanlar majmuasida yo'q 14 ta yo'nalish (asosan 2026 yilda yangi ochilganlar) uchun fanlar `references/kirish_imtihon_fanlari.md` dagi jadvallardan olinadi:

- `Fizika (yoki Chet tili)` kabi muqobil variantlarda asosiy fan (qavsdan oldingi) olinadi.
- Fan nomlari abituriyentlar jadvalidagi nomlarga keltiriladi (`Huquqshunoslik` → `Huquqshunoslik fanlari`).

In [10]:
MD_FAN_NOMI = {'Huquqshunoslik': 'Huquqshunoslik fanlari'}

def md_jadvallarni_oqi(yol):
    qatorlar = []
    with open(yol, encoding='utf-8') as f:
        for satr in f:
            ustunlar = [u.strip() for u in satr.strip().strip('|').split('|')]
            if not satr.startswith('|') or len(ustunlar) < 3:
                continue
            if ustunlar[0].startswith(':') or ustunlar[0] == "Yo'nalish nomi":
                continue
            qatorlar.append(ustunlar[:3])
    jadval = pd.DataFrame(qatorlar, columns=['nom', '1-fan', '2-fan'])
    jadval['nom'] = jadval['nom'].str.replace('**', '', regex=False)
    for col in ['1-fan', '2-fan']:
        jadval[col] = (jadval[col].str.replace(r'\s*\(.*\)', '', regex=True).str.strip()
                                  .replace(MD_FAN_NOMI))
    return jadval

qoshimcha = md_jadvallarni_oqi('../references/kirish_imtihon_fanlari.md')
qoshimcha['kalit'] = qoshimcha['nom'].map(normallash)
assert qoshimcha['kalit'].is_unique
assert not qoshimcha['kalit'].isin(fanlar['kalit']).any(), 'fanlar majmuasi bilan ustma-ust tushdi'

# Barcha fan nomlari abituriyentlar jadvalidagi nomlar bilan bir xil bo'lishi kerak
abt_fanlar = set(pd.read_csv('../data/processed/abts_prepared.csv', usecols=['1-fan', '2-fan']).stack())
assert set(qoshimcha[['1-fan', '2-fan']].stack()) <= abt_fanlar
print('Qo\'shimcha manbadagi yo\'nalishlar:', len(qoshimcha))
qoshimcha[['nom', '1-fan', '2-fan']]

Qo'shimcha manbadagi yo'nalishlar: 14


,nom,1-fan,2-fan
0,Qayta tiklanuvchi energiya manbalari,Matematika,Fizika
1,"Qurilish materiallari, buyumlari va konstruksiyalarini ishlab chiqarish",Matematika,Fizika
2,Ishlab chiqarish muhandisligi,Matematika,Fizika
3,Sanoat muhandisligi va menejmenti,Matematika,Fizika
4,Barqaror transport,Matematika,Fizika
5,Agromuhandislik,Matematika,Fizika
6,Sun'iy intellekt va kiberxavfsizlik,Matematika,Fizika
7,Xalqaro huquq va qiyosiy huquqshunoslik,Huquqshunoslik fanlari,Chet tili
8,Kiberhuquq,Huquqshunoslik fanlari,Chet tili
9,Raqamli huquq,Huquqshunoslik fanlari,Chet tili


In [11]:
IMLO_TUZATISH = {
    "neft' va neft-gazni qayta ishlash texnologiyasi": 'neft va neft-gazni qayta ishlash texnologiyasi',
    'tuproq bonitirovkasi va yer degredatsiyasi': 'tuproq bonitirovkasi va yer degradatsiyasi',
    'ishlab chqarish muhandisligi': 'ishlab chiqarish muhandisligi',
}

juftlik_map = fanlar.set_index('kalit')[['1-fan', '2-fan']]
qoshimcha_map = qoshimcha.set_index('kalit')[['1-fan', '2-fan']]

# Asosiy qism (':' gacha) bo'yicha eng ko'p uchraydigan fanlar juftligi
fanlar['asosiy_qism'] = fanlar['kalit'].str.split(':').str[0].str.strip()
asosiy_map = (fanlar.groupby('asosiy_qism')[['1-fan', '2-fan']]
                    .agg(lambda s: s.mode().iloc[0]))

def fanlarni_top(nom):
    kalit = normallash(nom)
    kalit = IMLO_TUZATISH.get(kalit, kalit)
    if kalit in juftlik_map.index:
        return (*juftlik_map.loc[kalit], 'aniq')
    if kalit in qoshimcha_map.index:
        return (*qoshimcha_map.loc[kalit], 'qoshimcha_manba')
    asosiy = kalit.split(':')[0].strip()
    if ':' in kalit and asosiy in asosiy_map.index:
        return (*asosiy_map.loc[asosiy], 'asosiy_qism')
    return (np.nan, np.nan, 'topilmadi')

natija = df['yunalish_nomi'].map(fanlarni_top)
df['1-fan'] = natija.str[0]
df['2-fan'] = natija.str[1]
df['fan_moslik'] = natija.str[2]

df['fan_moslik'].value_counts()

fan_moslik
aniq               4892
asosiy_qism         595
qoshimcha_manba      60
Name: count, dtype: int64

In [12]:
# Asosiy qism bo'yicha topilganlar — tekshirish uchun
(df[df['fan_moslik'] == 'asosiy_qism']
   .groupby(['asosiy_yunalish', '1-fan', '2-fan']).size()
   .reset_index(name='qatorlar')
   .sort_values('asosiy_yunalish')
   .query("`1-fan` != 'Kasbiy (ijodiy imtihon)'"))

,asosiy_yunalish,1-fan,2-fan,qatorlar
32,Filologiya va tillarni oʻqitish: pushtu tili,Chet tili,Ona tili va adabiyoti,2
114,Tarjima nazariyasi va amaliyoti: hind tili,Chet tili,Ona tili va adabiyoti,1
115,Tarjima nazariyasi va amaliyoti: ispan tili,Chet tili,Ona tili va adabiyoti,2
116,Tarjima nazariyasi va amaliyoti: urdu tili,Chet tili,Ona tili va adabiyoti,1
128,Xorijiy til va adabiyoti: xitoy tili,Chet tili,Ona tili va adabiyoti,3


In [13]:
# Qo'shimcha manbadan topilganlar
(df[df['fan_moslik'] == 'qoshimcha_manba']
   .groupby(['asosiy_yunalish', '1-fan', '2-fan']).size()
   .reset_index(name='qatorlar'))

,asosiy_yunalish,1-fan,2-fan,qatorlar
0,Agromuhandislik,Matematika,Fizika,3
1,Barqaror transport,Matematika,Fizika,4
2,Davlat auditi,Matematika,Chet tili,3
3,Davlat auditi talabiga mos emas,Matematika,Chet tili,3
4,Ishlab chqarish muhandisligi,Matematika,Fizika,6
5,Islom iqtisodiyoti va moliyasi,Matematika,Chet tili,1
6,Islomiy moliya,Matematika,Chet tili,1
7,Kiberhuquq,Huquqshunoslik fanlari,Chet tili,2
8,Qayta tiklanuvchi energiya manbalari,Matematika,Fizika,18
9,"Qurilish materiallari, buyumlari va konstruksiyalarini ishlab chiqarish",Matematika,Fizika,8


### Topilmagan yo'nalishlar

Barcha manbalardan keyin ham fanlari topilmagan yo'nalishlar — bo'sh bo'lishi kerak.

In [14]:
(df[df['fan_moslik'] == 'topilmadi']
   .groupby('asosiy_yunalish')
   .agg(qatorlar=('otm_nomi', 'size'), otm_lar=('otm_nomi', 'nunique'),
        grant_kvota=('grant_kvota_soni', 'sum'), shartnoma_kvota=('shartnoma_kvota_soni', 'sum'))
   .sort_values('qatorlar', ascending=False))

,qatorlar,otm_lar,grant_kvota,shartnoma_kvota
asosiy_yunalish,,,,


In [15]:
df['fanlar_juftligi'] = df['1-fan'] + ' + ' + df['2-fan']
print('Noyob fanlar juftligi:', df['fanlar_juftligi'].nunique())

# Abituriyentlar jadvalidagi juftliklar bilan solishtiramiz
abts = pd.read_csv('../data/processed/abts_prepared.csv', usecols=['fanlar_juftligi'])
otm_juft = set(df['fanlar_juftligi'].dropna())
abt_juft = set(abts['fanlar_juftligi'])
print('Faqat OTM jadvalida:         ', sorted(otm_juft - abt_juft))
print('Faqat abituriyentlar jadvalida:', sorted(abt_juft - otm_juft))

Noyob fanlar juftligi: 39


Faqat OTM jadvalida:          ['Qirgʻiz tili va adabiyoti + Tarix']
Faqat abituriyentlar jadvalida: []


## 7. Yakuniy tekshiruv va saqlash

In [16]:
df = df[['hudud', 'otm_nomi', 'yunalish_nomi', 'asosiy_yunalish', 'tuman_kvotasi', 'tuman',
         'talim_tili', 'talim_shakli', '1-fan', '2-fan', 'fanlar_juftligi', 'fan_moslik',
         'grant_kvota_soni', 'shartnoma_kvota_soni', 'grant_bali', 'shartnoma_bali']]
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5547 entries, 0 to 5546
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   hudud                 5547 non-null   str    
 1   otm_nomi              5547 non-null   str    
 2   yunalish_nomi         5547 non-null   str    
 3   asosiy_yunalish       5547 non-null   str    
 4   tuman_kvotasi         5547 non-null   bool   
 5   tuman                 724 non-null    str    
 6   talim_tili            5547 non-null   str    
 7   talim_shakli          5547 non-null   str    
 8   1-fan                 5547 non-null   object 
 9   2-fan                 5547 non-null   object 
 10  fanlar_juftligi       5547 non-null   object 
 11  fan_moslik            5547 non-null   object 
 12  grant_kvota_soni      5547 non-null   int64  
 13  shartnoma_kvota_soni  5547 non-null   int64  
 14  grant_bali            3376 non-null   float64
 15  shartnoma_bali        4718 non-n

In [17]:
assert not df.duplicated(KALIT).any()
assert ((df['grant_kvota_soni'] > 0) | (df['shartnoma_kvota_soni'] > 0)).all()
assert (df['grant_bali'].isna() == (df['grant_kvota_soni'] == 0)).all()
assert (df['shartnoma_bali'].isna() == (df['shartnoma_kvota_soni'] == 0)).all()
assert (df['grant_bali'] >= df['shartnoma_bali']).where(df[['grant_bali', 'shartnoma_bali']].notna().all(axis=1), True).all()
assert df[['1-fan', '2-fan', 'fanlar_juftligi']].notna().all().all(), 'fanlari topilmagan yo\'nalish bor'
print('Barcha tekshiruvlardan o\'tdi:', df.shape)

Barcha tekshiruvlardan o'tdi: (5547, 16)


In [18]:
df.to_csv('../data/processed/otm_prepared.csv', index=False, encoding='utf-8')
print('Saqlandi: data/processed/otm_prepared.csv')

Saqlandi: data/processed/otm_prepared.csv


## Natija

`otm_prepared.csv` — **5 547 qator, 16 ustun** (5 712 dan: 13 to'liq takror, 1 kalit to'qnashuvi birlashtirildi, 151 kvotasiz qator olib tashlandi).

- O'tish ballarida 0 yo'q: grant bali 3 376 qatorda, shartnoma bali 4 718 qatorda bor, qolgani NaN (kvota 0). Grant bali hamma joyda shartnoma balidan past emas.
- 724 qator tuman maqsadli kvotasi (183 ta tuman/shahar). Qavsni olib tashlaganda 341 ta asosiy yo'nalish.
- Fanlar **barcha qatorlarga** biriktirildi: fanlar majmuasidan aniq moslik, `kirish_imtihon_fanlari.md` dan 14 ta yangi yo'nalish, qolgan ixtisosliklar asosiy qism bo'yicha (`fan_moslik` ustunida manbasi).
- OTM jadvalidagi fanlar juftliklaridan faqat `Qirgʻiz tili va adabiyoti + Tarix` abituriyentlar jadvalida yo'q.

### Tasdiqlangan qarorlar
1. `kirish_imtihon_fanlari.md` dagi muqobil variantlarda asosiy fan olinadi: `Sanoat muhandisligi va menejmenti` va `Sunʼiy intellekt va kiberxavfsizlik` uchun 2-fan — Fizika.
2. `Davlat auditi talabiga mos emas` — alohida yo'nalish, shunday qoldiriladi.
3. TDIU `Davlat auditi` (O'zbekcha, Kunduzgi) ikki qatorining kvotalari qo'shiladi (150 + 9).